# Step 8: evaluate one or more fine-tuned adapters on dev (prompt v2)

Loads base Qwen2.5-Coder-1.5B-Instruct in fp16 **once**, then for each adapter listed in
`ADAPTERS_TO_EVALUATE`: downloads it from a private Hugging Face repo, attaches it to a
**freshly-reloaded copy** of the base model (merging mutates weights in place, so each
adapter needs its own copy, not a shared one), and runs it on all 1,034 dev prompts (prompt
v2 -- the Step 4+5 winner), using the **same decoding settings as the baseline** (greedy,
`max_new_tokens=256`, stop at `<|im_end|>`) so every comparison is fair.

Evaluating several checkpoints from the same run in one session (not just the final
adapter) is how Run B compares training length as well as LoRA capacity: Run A's
validation loss was actually lowest at step 63 (1/4 of the epoch), not at the end, so Run
B's training notebook pushes an adapter snapshot at every eval step (63, 126, 189, 252),
and this notebook evaluates all of them on dev EX, not just the final one. See PROGRESS.md.

Inference runs in fp16, not 4-bit: the base model is only 1.5B parameters (fits easily in a
T4's 16GB at fp16), and evaluating the merged adapter at full precision avoids any
quantization-specific generation differences between this and the fp16 baseline run --
QLoRA's 4-bit loading was a *training*-time memory trick (Step 7), not a requirement for
inference. This is a real, if small, train/inference precision difference (the adapter was
trained against a 4-bit quantized base and is evaluated merged into the fp16 base) --
standard QLoRA practice, and recorded explicitly in `run_metadata.json` rather than left
implicit.

**Before Run All:**
- Settings -> Accelerator: **GPU T4 x2** (only one T4 used, intentional), Internet: **On**.
- Add Input -> the prompts dataset matching `SPLIT` below (`prompts_v2_dev.jsonl`, the same
  one uploaded for Step 4+5, or `prompts_v2_test.jsonl` for the one final test run).
- Add-ons -> Secrets: **`HF_TOKEN`** (read access is enough -- it only downloads adapters
  from a private repo here, doesn't push anything).
- `SPLIT` in the config cell defaults to `"dev"`. Only set it to `"test"` for the one final
  chosen configuration (CLAUDE.md's dev-vs-test rule) -- never for iteration.
- Edit `ADAPTERS_TO_EVALUATE` in the config cell to the list of adapters you want evaluated
  this run -- the default below is Run B's step-63/126/189 snapshots plus its final adapter
  (step 252 is the final adapter, so it isn't listed twice).

**What this notebook does:**
- Same environment guard, `safe_pip_install` dry-run-before-install pattern, `torchao`
  uninstall, and `peft` preflight as the training notebooks (never risk the on-disk
  torch/CUDA build -- see the vLLM lesson in PROGRESS.md).
- For each adapter: downloads it, checks its `adapter_config.json`'s
  `base_model_name_or_path` is in `ACCEPTED_ADAPTER_BASE_NAMES` (an allow-list, not a single
  exact match -- Unsloth loads its own pre-quantized repo during training, e.g.
  `unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit`, not the original `Qwen/...` repo, even
  though it's the same underlying model) before loading it, fails loudly on anything not on
  the list, attaches it to a **freshly-reloaded** base model with PEFT, and merges it.
- Batched HF `generate`, fp16, left-padded, greedy decoding, `max_new_tokens=256`, stops at
  the model's real end-of-turn token (`<|im_end|>`), length-sorted batching restored to
  original order, automatic batch-size halving on a CUDA OOM -- identical generation code to
  the Step 4+5 baseline notebook, so any accuracy difference comes from the adapter, not
  from a different decoding setup. The tokenizer (and its stop-token resolution) is loaded
  **once** and shared across every adapter -- it doesn't change between adapters.
- **Resumable per adapter:** if a given adapter's `predictions_<tag>.jsonl` is already
  complete (from a Notebook Output Files input of an earlier version), that adapter is
  skipped entirely -- not even downloaded -- before moving to the next one in the list. A
  partial file is continued from where it stopped.
- Output: one `/kaggle/working/predictions_<tag>.jsonl` per adapter (same schema as the
  baseline's predictions files -- idx, db_id, question, raw_output, gen_time_sec -- so
  `scripts/evaluate_experiment.py` works on each one unmodified) and one combined
  `run_metadata.json` with a per-adapter breakdown.

**IMPORTANT -- use Save Version -> Save & Run All, not an interactive Run All.** An
interactive-only run's `/kaggle/working` is wiped when the session ends, which is exactly
what happened to Run A's training output (see PROGRESS.md). Only a saved Version's output
files survive and can be downloaded or attached later.

## How to resume an interrupted run
1. Make sure the run you want to resume from was saved with **Save Version -> Save & Run All**.
2. Open this notebook again (new session is fine).
3. **Add Input** -> **Notebook Output Files** tab -> find this notebook -> pick the version
   to resume from -> **Add**. Keep the prompts dataset attached too.
4. **Run All** again. For each adapter in `ADAPTERS_TO_EVALUATE`, the notebook checks for a
   `predictions_<tag>.jsonl` under `/kaggle/input/...`: a complete one is skipped entirely
   (adapter not even re-downloaded), a partial one is copied in and continued from the first
   row that hasn't been generated yet.


In [ ]:
# Environment check -- catch a broken environment before spending any GPU time.
import json as _json
import os
import subprocess
import sys

print("Python:", sys.version)

import torch
print("torch:", torch.__version__, "| torch's CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available in this session. Check Settings -> Accelerator is set to GPU T4 x2, "
        "then Run All again."
    )

print("GPU:", torch.cuda.get_device_name(0))

import transformers
print("transformers:", transformers.__version__)

# Packages a pip install must never be allowed to touch -- this is exactly what broke an
# earlier notebook's vLLM install (pip silently upgraded torchaudio to a build that didn't
# match the already-loaded torch).
WATCHED_PACKAGES = ["torch", "torchvision", "torchaudio", "triton", "transformers"]


def installed_torch_build():
    """What's actually on disk right now, read via a FRESH subprocess -- not the
    already-imported `torch` module in this kernel, which can't reflect a pip install
    that happened after it was first imported."""
    r = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.version.cuda)"],
        capture_output=True, text=True,
    )
    lines = r.stdout.strip().splitlines()
    return (lines[0], lines[1]) if len(lines) >= 2 else (None, None)


TORCH_BUILD_AT_START = installed_torch_build()
print("torch build on disk, recorded for later comparison:", TORCH_BUILD_AT_START)


def _current_versions(names):
    out = {}
    for name in names:
        r = subprocess.run(
            [sys.executable, "-c", f"import importlib.metadata as m; print(m.version('{name}'))"],
            capture_output=True, text=True,
        )
        out[name] = r.stdout.strip() if r.returncode == 0 else None
    return out


def dry_run_would_change_watched(args):
    """Simulate the install with `pip install --dry-run --report <file>` -- installs
    nothing -- and check whether the packages it WOULD install/upgrade include any of
    WATCHED_PACKAGES at a version different from what's currently on disk. If anything
    would change, the real install never runs at all."""
    report_path = os.path.join(OUTPUT_DIR, "_pip_dry_run_report.json")
    if os.path.exists(report_path):
        os.remove(report_path)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--dry-run", "--quiet",
         "--report", report_path] + args,
        capture_output=True, text=True,
    )
    if result.returncode != 0 or not os.path.exists(report_path):
        return True, {"error": (result.stderr or "no dry-run report produced").strip()[:500]}

    with open(report_path, encoding="utf-8") as f:
        report = _json.load(f)

    current = _current_versions(WATCHED_PACKAGES)
    changes = {}
    for item in report.get("install", []):
        meta = item.get("metadata", {})
        pkg_name = meta.get("name", "")
        pkg_version = meta.get("version")
        for watched in WATCHED_PACKAGES:
            if pkg_name.lower() == watched.lower() and pkg_version != current.get(watched):
                changes[watched] = {"current": current.get(watched), "would_become": pkg_version}
    return (len(changes) > 0), changes


def safe_pip_install(args, label):
    """Install `args`, but only after a dry run confirms it wouldn't touch any
    WATCHED_PACKAGES. Returns True if installed (or already satisfied), False if skipped."""
    would_change, details = dry_run_would_change_watched(args)
    if would_change:
        print(f"{label}: SKIPPING install -- a dry run shows it would change a protected "
              f"package: {details}. Nothing was installed.")
        return False

    before = installed_torch_build()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args, check=True)
    after = installed_torch_build()
    if after != before:
        raise RuntimeError(
            f"{label}: installing it changed the on-disk torch build from {before} to "
            f"{after} even though the dry-run check passed. Stopping the notebook rather "
            f"than continue on a possibly broken environment."
        )
    return True


In [ ]:
# ---- Config -- edit ADAPTERS_TO_EVALUATE for what this session should evaluate ----
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 on purpose

OUTPUT_DIR = OUT_DIR = "/kaggle/working"

BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# "dev" for all normal iteration. "test" is Spider test (2,147 questions) -- run it only
# for the one final chosen configuration, exactly once, per CLAUDE.md's dev-vs-test rule.
# Flipping this also changes which prompts file this notebook looks for (below) and which
# split scripts/evaluate_experiment.py should be pointed at afterward.
SPLIT = "dev"
assert SPLIT in ("dev", "test"), f"SPLIT must be 'dev' or 'test', got {SPLIT!r}"

# adapter_config.json's base_model_name_or_path is whatever repo the training notebook
# actually loaded weights from -- Unsloth loads its own pre-quantized copy of the base
# model rather than the original Qwen repo, so an Unsloth-trained adapter_config.json
# records "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit", not "Qwen/Qwen2.5-Coder-1.5B-
# Instruct", even though it's the same underlying model and the base weights loaded here
# (fp16, from BASE_MODEL_NAME above) are the real Qwen ones. An allow-list, not a single
# exact match, is what actually checks "is this the right model" -- a bare != check would
# reject every Unsloth-trained adapter even though they're correct.
ACCEPTED_ADAPTER_BASE_NAMES = [
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
]

# Each entry: {"repo": <HF repo id>, "path_in_repo": <folder in that repo>, "tag": <short
# name>}. "tag" becomes predictions_<tag>.jsonl. Evaluated in order, in one session --
# edit this list for what you want evaluated this run.
# "repo": None (with "path_in_repo": None too) means the plain base model, no adapter --
# use it to score BASE_MODEL_NAME itself in the same session as the adapters, with
# identical decoding settings, e.g. for a fair base-vs-fine-tuned test-set comparison.
# adapter_step_252 is left out on purpose: step 252 is the last step, so it is the same
# weights as final_adapter, and evaluating both would just repeat the same run.
ADAPTERS_TO_EVALUATE = [
    {"repo": "FARHANFAST/text2sql-qlora-run-b", "path_in_repo": "adapter_step_63", "tag": "run_b_step63"},
    {"repo": "FARHANFAST/text2sql-qlora-run-b", "path_in_repo": "adapter_step_126", "tag": "run_b_step126"},
    {"repo": "FARHANFAST/text2sql-qlora-run-b", "path_in_repo": "adapter_step_189", "tag": "run_b_step189"},
    {"repo": "FARHANFAST/text2sql-qlora-run-b", "path_in_repo": "final_adapter", "tag": "run_b_final"},
]
for _entry in ADAPTERS_TO_EVALUATE:
    assert _entry["repo"] is None or "CHANGE_ME" not in _entry["repo"], (
        f"Edit the repo for tag {_entry['tag']!r} in ADAPTERS_TO_EVALUATE before running."
    )
assert len(set(_e["tag"] for _e in ADAPTERS_TO_EVALUATE)) == len(ADAPTERS_TO_EVALUATE), (
    "Duplicate tag in ADAPTERS_TO_EVALUATE -- each adapter needs its own unique tag, since "
    "it becomes the output filename."
)

SEED = 0
MAX_NEW_TOKENS = 256
MAX_MODEL_LEN = 4096      # matches the Step 7 training cutoff
CHUNK_SIZE = 32            # examples per flush/resume checkpoint
INITIAL_BATCH_SIZE = 8     # halves automatically and retries on CUDA OOM


In [ ]:
# Kaggle's preinstalled torchao (0.10.0) conflicts with the peft version this notebook
# needs -- confirmed on the real Run A evaluation run. Uninstall it up front, before the
# peft preflight cell below. A REMOVAL, not an install pip could resolve into touching
# torch/torchvision/torchaudio, so it doesn't go through the dry-run safety check -- but
# the on-disk torch build is still checked unchanged afterward, same as safe_pip_install.
import importlib.metadata as _im

try:
    _torchao_version = _im.version("torchao")
    _before_torchao_uninstall = installed_torch_build()
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    _after_torchao_uninstall = installed_torch_build()
    if _after_torchao_uninstall != _before_torchao_uninstall:
        raise RuntimeError(
            f"uninstalling torchao changed the on-disk torch build from "
            f"{_before_torchao_uninstall} to {_after_torchao_uninstall} -- stopping rather "
            f"than continue on a possibly broken environment."
        )
    print(f"uninstalled torchao {_torchao_version} (was conflicting with the peft version "
          f"this notebook needs)")
except _im.PackageNotFoundError:
    print("torchao not installed -- nothing to do")


In [ ]:
# peft preflight: make sure it imports cleanly and print its version before any adapter
# is attached below. Only installs it if actually missing, and only after confirming (dry
# run) it wouldn't touch a watched package.
try:
    import peft
    print("peft already installed:", peft.__version__)
except ImportError:
    ok = safe_pip_install(["peft"], "peft")
    if not ok:
        raise RuntimeError("peft is required to load the adapters and its install was "
                            "blocked by the dry-run safety check -- see the message above.")
    import peft
    print("peft installed:", peft.__version__)


In [ ]:
# Kaggle secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("Missing Kaggle secret HF_TOKEN. Add-ons -> Secrets -> add HF_TOKEN "
                        "(read access is enough), then Run All again.")
print("secret loaded OK")


In [ ]:
# Find the prompts file under /kaggle/input for SPLIT (recursive search, never a fixed
# dataset slug).
import glob


def find_input_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return matches[0] if matches else None


PROMPTS_FNAME = f"prompts_v2_{SPLIT}.jsonl"
PROMPTS_PATH = find_input_file(PROMPTS_FNAME)
if PROMPTS_PATH is None:
    raise FileNotFoundError(
        f"Missing {PROMPTS_FNAME} under /kaggle/input. Attach the matching prompts dataset "
        f"(or re-upload data/processed/kaggle_upload/{PROMPTS_FNAME})."
    )
print(f"found {PROMPTS_FNAME} -> {PROMPTS_PATH}")


In [ ]:
# Tokenizer loaded ONCE -- shared across every adapter below. It doesn't change between
# adapters (only the LoRA weights merged into the base model do).
from transformers import AutoTokenizer

torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for correct batched causal-LM generation

# Qwen's chat template ends a turn with <|im_end|>, which may not be the tokenizer's
# default eos_token_id -- stop generation at both, same as the Step 4+5 baseline.
eos_ids = {tokenizer.eos_token_id}
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if im_end_id is not None and im_end_id != getattr(tokenizer, "unk_token_id", None):
    eos_ids.add(im_end_id)
eos_ids = sorted(eos_ids)
print("stopping generation at token ids:", eos_ids, [tokenizer.decode([i]) for i in eos_ids])


In [ ]:
def generate_batch_safe(prompts, batch_size):
    """Generate for `prompts` against the CURRENT global `model`, halving batch_size and
    retrying on a CUDA OOM instead of crashing."""
    try:
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                        max_length=MAX_MODEL_LEN).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id)
        gen_only = out[:, enc["input_ids"].shape[1]:]
        return tokenizer.batch_decode(gen_only, skip_special_tokens=True)
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        torch.cuda.empty_cache()
        if batch_size <= 1:
            raise
        half = batch_size // 2
        print(f"  CUDA OOM at batch_size={batch_size}, halving to {half} and retrying")
        results = []
        for i in range(0, len(prompts), half):
            results.extend(generate_batch_safe(prompts[i:i + half], half))
        return results


def generate_chunk(rows, batch_size):
    """rows: list of {idx, db_id, question, prompt}. Returns raw output strings in the
    SAME order as `rows` -- sorts by prompt length internally, restores original order."""
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["prompt"]))
    sorted_prompts = [rows[i]["prompt"] for i in order]
    sorted_outputs = []
    for i in range(0, len(sorted_prompts), batch_size):
        batch = sorted_prompts[i:i + batch_size]
        sorted_outputs.extend(generate_batch_safe(batch, batch_size))
    outputs = [None] * len(rows)
    for pos, orig_idx in enumerate(order):
        outputs[orig_idx] = sorted_outputs[pos]
    return outputs


In [ ]:
import json
import shutil
import time

from huggingface_hub import HfApi, snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM


def load_prompts(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def load_done(path):
    done = {}
    if not os.path.exists(path):
        return done
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            done[row["idx"]] = row
    return done


def evaluate_one_adapter(entry, rows, hf_api):
    """Downloads, validates, attaches (to a fresh base model copy), and generates for one
    adapter -- or, if entry["repo"] is None, just the plain base model, no adapter at all.
    Resumable: if predictions_<tag>.jsonl is already complete (e.g. attached from a prior
    Notebook Output Files version), skips entirely without even downloading the adapter.
    Returns this adapter's metadata dict for run_metadata.json."""
    tag, adapter_repo, adapter_path_in_repo = entry["tag"], entry["repo"], entry["path_in_repo"]
    is_base_only = adapter_repo is None
    label = "plain base model, no adapter" if is_base_only else f"{adapter_repo}/{adapter_path_in_repo}"
    print(f"\n{'=' * 70}\n=== {tag}: {label} ===\n{'=' * 70}")

    out_path = os.path.join(OUT_DIR, f"predictions_{tag}.jsonl")
    done = load_done(out_path)
    if not done:
        prev_path = find_input_file(f"predictions_{tag}.jsonl")
        if prev_path is not None:
            shutil.copy(prev_path, out_path)
            done = load_done(out_path)
            if done:
                print(f"  found a previous output for {tag} -> copied {prev_path}")

    if len(done) >= len(rows):
        print(f"  {tag}: already complete ({len(done)}/{len(rows)}) -- skipping entirely, "
              f"not even downloading the adapter")
        return {"adapter_repo": adapter_repo, "adapter_path_in_repo": adapter_path_in_repo,
                "status": "already complete, skipped"}

    global model
    base_model = None  # stays None on the is_base_only path -- `del model, base_model` below needs it defined
    if is_base_only:
        # No adapter to download or validate -- just the base model, fp16, same as every
        # other entry's starting point. Still reloaded fresh (not reused) for consistency
        # with the adapter path, even though there's no merge to worry about carrying over.
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME, torch_dtype=torch.float16
        ).to("cuda")
        model.eval()
        adapter_config = None
        adapter_base = None
        print(f"  {tag}: base model loaded, no adapter attached")
    else:
        repo_info = hf_api.repo_info(repo_id=adapter_repo)
        print(f"  adapter repo {adapter_repo} visibility: "
              f"{'private' if repo_info.private else 'PUBLIC'}")
        adapter_local_dir = os.path.join(OUTPUT_DIR, "adapters", tag)
        snapshot_download(repo_id=adapter_repo, local_dir=adapter_local_dir,
                           allow_patterns=f"{adapter_path_in_repo}/*", token=HF_TOKEN)
        adapter_dir = os.path.join(adapter_local_dir, adapter_path_in_repo)

        with open(os.path.join(adapter_dir, "adapter_config.json"), encoding="utf-8") as f:
            adapter_config = json.load(f)
        adapter_base = adapter_config.get("base_model_name_or_path")
        if adapter_base not in ACCEPTED_ADAPTER_BASE_NAMES:
            raise ValueError(
                f"{tag}: adapter at {adapter_repo}/{adapter_path_in_repo} was trained against "
                f"base model '{adapter_base}', which isn't in ACCEPTED_ADAPTER_BASE_NAMES "
                f"{ACCEPTED_ADAPTER_BASE_NAMES} -- refusing to attach it to the wrong base."
            )
        print(f"  adapter base model '{adapter_base}' matched an accepted name")
        print(f"  adapter LoRA config: r={adapter_config.get('r')}, "
              f"alpha={adapter_config.get('lora_alpha')}")

        # Fresh base model copy for THIS adapter -- merge_and_unload mutates weights in
        # place, so a shared/reused base model would carry the previous adapter's merged
        # weights into the next one. Reloading from the (already-cached) Hub download is
        # cheap for a 1.5B model -- a few seconds, not worth a GPU-side deep copy instead.
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME, torch_dtype=torch.float16
        ).to("cuda")
        model = PeftModel.from_pretrained(base_model, adapter_dir)
        model = model.merge_and_unload()
        model.eval()
        print(f"  {tag}: base model + adapter loaded and merged")

    with open(out_path, "w", encoding="utf-8") as f:
        for idx in sorted(done):
            f.write(json.dumps(done[idx]) + "\n")
    remaining = [r for r in rows if r["idx"] not in done]
    if done:
        print(f"  {tag}: RESUMING -- {len(done)}/{len(rows)} already done, "
              f"{len(remaining)} remaining")
    else:
        print(f"  {tag}: STARTING FRESH -- {len(remaining)} to do")

    adapter_start = time.time()
    with open(out_path, "a", encoding="utf-8") as f:
        for i in range(0, len(remaining), CHUNK_SIZE):
            chunk = remaining[i:i + CHUNK_SIZE]
            t0 = time.time()
            raw_outputs = generate_chunk(chunk, INITIAL_BATCH_SIZE)
            elapsed = time.time() - t0
            per_example_time = elapsed / len(chunk)
            for r, raw in zip(chunk, raw_outputs):
                f.write(json.dumps({
                    "idx": r["idx"], "db_id": r["db_id"], "question": r["question"],
                    "raw_output": raw, "gen_time_sec": per_example_time,
                }) + "\n")
            f.flush()
            os.fsync(f.fileno())
            done_so_far = len(done) + i + len(chunk)
            print(f"  {tag}: {done_so_far}/{len(rows)} done, last chunk {elapsed:.1f}s")
    adapter_elapsed = time.time() - adapter_start
    print(f"  === {tag} done in {adapter_elapsed:.1f}s, wrote {out_path} ===")

    del model, base_model
    torch.cuda.empty_cache()

    return {
        "adapter_repo": adapter_repo, "adapter_path_in_repo": adapter_path_in_repo,
        "adapter_config": adapter_config,
        "base_model_adapter_trained_on": adapter_base,
        "train_inference_base_precision_note": (
            "This adapter's own adapter_config.json records the base model it was trained "
            "against as the training notebook's loaded copy (Unsloth loads its own "
            "pre-quantized 4-bit copy rather than the original Qwen repo, so this can read "
            "'unsloth/...-bnb-4bit' even for a plain Qwen base). This notebook evaluates the "
            "adapter merged into the full fp16 base model (BASE_MODEL_NAME), not the 4-bit "
            "quantized base it trained against. Standard QLoRA practice -- the LoRA delta is "
            "small relative to full fp16 precision -- but a real train/inference precision "
            "difference, not nothing, recorded here rather than left implicit."
        ),
        "n_examples": len(rows), "newly_generated": len(remaining),
        "total_time_sec": adapter_elapsed,
    }


In [ ]:
# Evaluate every adapter in ADAPTERS_TO_EVALUATE, in order, in this one session.
api = HfApi(token=HF_TOKEN)
rows = load_prompts(PROMPTS_PATH)

run_metadata = {
    "base_model_evaluated_on": BASE_MODEL_NAME,
    "split": SPLIT, "prompt_version": "v2", "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy", "eos_token_ids": eos_ids,
    "python_version": sys.version, "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda, "transformers_version": transformers.__version__,
    "peft_version": peft.__version__, "gpu_name": torch.cuda.get_device_name(0),
    "adapters": {},
}

for entry in ADAPTERS_TO_EVALUATE:
    run_metadata["adapters"][entry["tag"]] = evaluate_one_adapter(entry, rows, api)

with open(os.path.join(OUT_DIR, "run_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
print(f"\nsaved run_metadata.json for {len(run_metadata['adapters'])} adapters")
for tag, meta in run_metadata["adapters"].items():
    print(f"  {tag}: {meta.get('status', 'generated ' + str(meta.get('newly_generated')) + ' rows')}")


## Download from /kaggle/working/ when done:
- `predictions_<tag>.jsonl` for every tag in `ADAPTERS_TO_EVALUATE`
- `run_metadata.json`

Put each `predictions_<tag>.jsonl` at `results/step8_<tag>/predictions_<tag>.jsonl` locally,
then run, per tag -- `--split` defaults to `dev`, so only pass it if `SPLIT` above was set
to `"test"` (plain EX only there: the test-suite perturbation databases only exist for dev):

```
python scripts/evaluate_experiment.py --pred results/step8_<tag>/predictions_<tag>.jsonl --tag <tag> [--split test]
```

If you need to stop partway through, use **Save Version -> Save & Run All** before
stopping, so the partial output is attachable later (see "How to resume" above). A
fully-finished adapter's `predictions_<tag>.jsonl` is skipped entirely on the next run --
only in-progress or not-yet-started adapters continue.